In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="my-dissertation-project-502620")

In [ ]:
query = """
SELECT
    c.subject_id,
    c.hadm_id,
    r.charttime,
    r.text
FROM `my-dissertation-project-502620.Query_Results.final_100_evaluation_cases` AS c
INNER JOIN `physionet-data.mimiciv_note.radiology` AS r
    ON c.subject_id = r.subject_id
    AND c.hadm_id = r.hadm_id
ORDER BY c.subject_id, c.hadm_id, r.charttime
"""

selected_radiology_reports = client.query(query).to_dataframe()

print("Number of radiology reports:", len(selected_radiology_reports))
print(
    "Number of admissions with radiology:",
    selected_radiology_reports["hadm_id"].nunique()
)

In [ ]:
from collections import Counter
import re
import pandas as pd

def analyse_radiology_sections(radiology_df):

    heading_pattern = re.compile(
        r"^\s*([A-Z][A-Z0-9 /,_().'-]*):\s*",
        re.MULTILINE
    )

    section_counter = Counter()
    examples = {}

    for _, row in radiology_df.iterrows():

        text = row["text"]

        if pd.isna(text):
            continue

        headings = heading_pattern.findall(text)

        headings = [
            re.sub(r"\s+", " ", heading).strip()
            for heading in headings
        ]

        # Count each heading once per report
        for heading in set(headings):
            section_counter[heading] += 1

            if heading not in examples:
                examples[heading] = {
                    "subject_id": row["subject_id"],
                    "hadm_id": row["hadm_id"],
                    "charttime": row["charttime"],
                    "text": text
                }

    summary = pd.DataFrame(
        section_counter.items(),
        columns=["section", "number_of_reports"]
    )

    summary = summary.sort_values(
        "number_of_reports",
        ascending=False
    ).reset_index(drop=True)

    return summary, examples

In [ ]:
section_summary, section_examples = analyse_radiology_sections(
    selected_radiology_reports
)

print(section_summary.to_string(index=False))

In [ ]:
import re
import pandas as pd

def analyse_findings_impression_coverage(radiology_df):

    def has_section(text, section):
        if pd.isna(text):
            return False

        pattern = rf"(?m)^\s*{re.escape(section)}\s*:"
        return bool(re.search(pattern, text, re.IGNORECASE))

    df = radiology_df.copy()

    df["has_findings"] = df["text"].apply(
        lambda x: has_section(x, "FINDINGS")
    )

    df["has_impression"] = df["text"].apply(
        lambda x: has_section(x, "IMPRESSION")
    )

    summary = pd.Series({
        "FINDINGS + IMPRESSION":
            ((df["has_findings"]) &
             (df["has_impression"])).sum(),

        "FINDINGS only":
            ((df["has_findings"]) &
             (~df["has_impression"])).sum(),

        "IMPRESSION only":
            ((~df["has_findings"]) &
             (df["has_impression"])).sum(),

        "Neither":
            ((~df["has_findings"]) &
             (~df["has_impression"])).sum(),

        "Total reports":
            len(df)
    })

    return summary, df

In [ ]:
coverage_summary, radiology_coverage = \
    analyse_findings_impression_coverage(selected_radiology_reports)

print(coverage_summary)

In [ ]:
# Select reports that have IMPRESSION but no FINDINGS
impression_only = radiology_coverage[
    (~radiology_coverage["has_findings"]) &
    (radiology_coverage["has_impression"])
].copy()

print("Total IMPRESSION-only reports:", len(impression_only))

# Random sample of 25
impression_only_sample = impression_only.sample(
    n=min(25, len(impression_only)),
    random_state=42
)

for i, (_, row) in enumerate(impression_only_sample.iterrows(), start=1):
    print("=" * 100)
    print(f"REPORT {i}")
    print(
        f"subject_id: {row['subject_id']} | "
        f"hadm_id: {row['hadm_id']} | "
        f"charttime: {row['charttime']}"
    )
    print("-" * 100)
    print(row["text"])
    print()

In [ ]:
import re

KNOWN_RAD_SECTIONS = [
    "EXAMINATION",
    "INDICATION",
    "CLINICAL INDICATION",
    "REASON",
    "REASON FOR EXAM",
    "REASON FOR EXAMINATION",
    "HISTORY",
    "CLINICAL HISTORY",
    "MEDICAL HISTORY",
    "CLINICAL INFORMATION",
    "COMPARISON",
    "COMPARISONS",
    "TECHNIQUE",
    "PROCEDURE",
    "PROCEDURE DETAILS",
    "FINDINGS",
    "IMPRESSION",
    "CONCLUSION",
    "COMMENT",
    "RECOMMENDATION",
    "RECOMMENDATIONS",
    "NOTIFICATION",
    "MEDICATIONS",
    "CONTRAST",
    "DOSE",
    "FLUOROSCOPY TIME AND DOSE",
    "DATE",
    "REFERENCE"
]


def parse_report(report):
    """
    Split a radiology report using recognised section headings.

    Handles both:
        FINDINGS:
        some text

    and:
        FINDINGS: some text
    """

    if not report or not isinstance(report, str):
        return {}

    # Longest headings first
    headings = sorted(KNOWN_RAD_SECTIONS, key=len, reverse=True)

    heading_pattern = "|".join(
        re.escape(h) for h in headings
    )

    # Heading must begin a line, but content may follow the colon
    pattern = re.compile(
        rf"(?im)^[ \t]*({heading_pattern})[ \t]*:"
    )

    matches = list(pattern.finditer(report))

    if not matches:
        return {"UNSECTIONED": report.strip()}

    sections = {}

    # Anything before the first recognised heading
    preamble = report[:matches[0].start()].strip()
    if preamble:
        sections["PREAMBLE"] = preamble

    for i, match in enumerate(matches):

        heading = match.group(1).upper()

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(report)

        content = report[start:end].strip()

        if content:
            # Avoid silently overwriting repeated headings
            if heading in sections:
                sections[heading] += "\n" + content
            else:
                sections[heading] = content

    return sections

In [ ]:
# test_radiology_reports = selected_radiology_reports[
#     (selected_radiology_reports["subject_id"] == 00000000) &
#     (selected_radiology_reports["hadm_id"] == 00000000)
# ].copy()

# print("Number of reports:", len(test_radiology_reports))



# for i, row in selected_radiology_reports.iterrows():

#     print("=" * 100)
#     print("CHARTTIME:", row["charttime"])

#     sections = parse_report(row["text"])

#     for heading, content in sections.items():
#         print(f"\n--- {heading} ---")
#         print(content)

#     print()



for i, row in selected_radiology_reports.iterrows():

    sections = parse_report(row["text"])

    print(
        row["subject_id"],
        row["hadm_id"],
        row["charttime"],
        "->",
        list(sections.keys())
    )